# Leakage-Safe Water Quality Index Forecasting

This notebook forecasts a river station's **next-year Water Quality Index (WQI)** from measurements available in the current year: pH, dissolved oxygen (DO), biochemical oxygen demand (BOD), total and fecal coliform counts, and nitrate. It uses the reusable implementation in `src/model.py`; no modeling logic is duplicated here.

Run this notebook from the repository root after `uv sync --group dev`.

## Research-paper context and project extension

The included paper (`paper/s12145-025-01865-2.pdf`) evaluates individual XGBR, RFR, SVR, and RNN models on 432 **monthly** Haridwar observations from 2017–2022, using an 80:20 split. Its reported R² values are 0.9128 (XGBR), 0.8927 (RFR), 0.9246 (SVR), and 0.9655 (RNN).

This repository contains 2003–2014 nationwide observations rather than that Haridwar dataset. LightGBM and the stacked ensemble are therefore project extensions. Paper scores are context for interpretation, not a target to optimize against.

In [1]:
import os
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport

DATA_PATH = Path('data/dataset.csv')
ASSETS_DIR = 'assets'

# Ensure assets dir exists
os.makedirs(ASSETS_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

profile = ProfileReport(df, title="Dataset Profiling Report", explorative=True)

output_path = os.path.join(ASSETS_DIR, "profile_report.html")
profile.to_file(output_path)

print(f"Report saved to: {output_path}")

ModuleNotFoundError: No module named 'pkg_resources'

## Why this is a forecast, not same-row fitting

WQI is calculated from water-quality measurements. Training on a WQI calculated from the *same row's* measurements would leak the answer into the inputs and can produce misleadingly high R². Instead, this notebook pairs a station's measurements at year *t* with its calculated WQI at year *t + 1*. Only consecutive-year station records are used.

Duplicate station/year rows are averaged. The documented five-input WQI approximation applies weights DO 0.31, pH 0.11, BOD 0.19, fecal coliform 0.28, and nitrate 0.11.

In [ ]:
from pathlib import Path

import pandas as pd

from src.model import (
    FEATURE_COLUMNS,
    TEST_YEAR,
    TRAIN_END_YEAR,
    VALIDATION_YEAR,
    aggregate_station_years,
    apply_feature_engineering,
    build_forecast_pairs,
    generate_shap_summary,
    load_dataset,
    print_results,
    run_pipeline,
    split_by_target_year,
)

DATA_PATH = Path('data/dataset.csv')

## Prepare station-year forecast pairs

The target year controls every split: target years through 2012 form training, 2013 forms validation, and 2014 is the untouched final test set. Numeric imputation and scaling are fit inside the training flow only.

In [ ]:
station_years = aggregate_station_years(load_dataset(DATA_PATH))
forecast_pairs = build_forecast_pairs(apply_feature_engineering(station_years))
partitions = split_by_target_year(forecast_pairs)

pd.DataFrame(
    {
        'partition': partitions.keys(),
        'target-year rule': [
            f'<= {TRAIN_END_YEAR}',
            str(VALIDATION_YEAR),
            str(TEST_YEAR),
        ],
        'forecast pairs': [len(partition) for partition in partitions.values()],
    }
)

## Benchmark models and generalization checks

The benchmark evaluates median and persistence baselines, Random Forest, XGBoost, LightGBM, SVR, and a Ridge meta-model stacked from chronological out-of-fold base-model predictions. The stack never uses random cross-validation or test labels.

Metrics are R², MAE, RMSE, and MSE. A train-to-validation R² drop above 0.10 is reported as an overfitting warning; model choices should be made from validation metrics before reading the final test results.

In [ ]:
results = run_pipeline(DATA_PATH)
print_results(results)

metric_table = pd.concat(
    {partition: pd.DataFrame(metrics).T for partition, metrics in results.items()},
    names=['partition', 'model'],
).round(4)
metric_table

## SHAP explainability

SHAP explains the fitted Random Forest on held-out test samples. It describes feature influence in this model; it does not establish causation. The output path is explicit so generated artifacts stay outside the repository unless intentionally saved elsewhere.

In [ ]:
shap_output = Path('/tmp/wqi-random-forest-shap.png')
generate_shap_summary(DATA_PATH, shap_output)
shap_output

## RNN: deferred until sequence validation

The research paper's RNN is appropriate for its monthly time series. This data is annual with irregular station histories, so an RNN is intentionally not included in the current benchmark. Implement it only after choosing and testing a sequence length, handling missing years, and ensuring all historical windows precede their targets. Install the optional dependency when needed with `uv sync --group rnn`.